# KAF-4 — Retention & Compaction

**Break → Detect → Fix → Prove.** A Kafka topic does not keep messages forever. Its
**`cleanup.policy`** decides what happens to old data: **`delete`** (default) drops records once
they age past **`retention.ms`** (or the log exceeds **`retention.bytes`**); **`compact`** keeps only
the **latest value per key**. Get it wrong and an offline consumer comes back to find its offset
**gone** (`OffsetOutOfRange`), or a changelog topic grows without bound.

**Pre-requisite:** `make up` (Kafka at `localhost:29092` for producers/admin, `kafka:9092` for
Spark). Inspect topic configs live in **kafka-ui** at http://localhost:8080 → topic → **Settings**.
**Laptop-safe:** small bounded produce batches and only short bounded `sleep`s — never an unbounded
wait on the broker; both topics are deleted at teardown.

> **Honesty about timing (read this).** Retention deletion and log compaction run on the **broker's
> own background schedule** (the log-cleaner / retention threads, gated by `segment.ms`,
> `min.cleanable.dirty.ratio`, `log.retention.check.interval.ms`). A client **cannot** force them to
> complete on demand. So this notebook proves what is **reliably observable now** — the *configs*
> are applied (`describe_configs`) and compaction's *semantics* (latest-value-per-key) — and is
> explicit that the *physical* deletion/collapse is **broker-scheduled** (we describe it and poll
> briefly, but never block). See the companion [`README.md`](./README.md).

In [ ]:
from common.kafka_helpers import (ensure_topic, produce_events, topic_end_offsets,
                                  delete_topic, BOOTSTRAP)
from kafka import KafkaConsumer, KafkaAdminClient
from kafka.admin import ConfigResource, ConfigResourceType
import time

EVENTS_TOPIC = "kaf4_events"   # cleanup.policy=delete  -> the offline-consumer / OffsetOutOfRange story
STATE_TOPIC  = "kaf4_state"    # cleanup.policy=compact -> the changelog / latest-per-key story
print("producers/admin ->", BOOTSTRAP, "| kafka-ui: http://localhost:8080")

## Step 0 — a tiny helper to read a topic's live config

`describe_configs` is our **proof tool**: it asks the broker what policy/retention/segment settings a
topic actually has. We use it throughout to show the configs are applied (independent of *when* the
broker's background threads next run).

In [ ]:
def topic_config(topic, keys=None):
    """Return selected (or all) live topic configs from the broker via describe_configs."""
    admin = KafkaAdminClient(bootstrap_servers=BOOTSTRAP)
    try:
        res = ConfigResource(ConfigResourceType.TOPIC, topic)
        described = admin.describe_configs([res])
        # kafka-python returns a list of responses; each has .resources -> [(err,errmsg,rtype,rname,[configs])]
        entries = described[0].resources[0][4]
        cfg = {c[0]: c[1] for c in entries}        # name -> value
        return cfg if keys is None else {k: cfg.get(k) for k in keys}
    finally:
        admin.close()

SHOW = ["cleanup.policy", "retention.ms", "retention.bytes", "segment.ms",
        "min.cleanable.dirty.ratio", "delete.retention.ms"]
print("helper ready")

## 1. Break it — (A) `delete` policy & the offline-consumer trap

We create `kaf4_events` with the default **`cleanup.policy=delete`** but a deliberately **tiny
`retention.ms`** and **tiny `segment.ms`**. The small `segment.ms` matters: only a **rolled**
(non-active) segment is eligible for deletion, so a fast-rolling segment lets retention actually bite.
We produce a bounded batch and record the offset range.

In [ ]:
ensure_topic(EVENTS_TOPIC, num_partitions=1, recreate=True, configs={
    "cleanup.policy": "delete",
    "retention.ms": "60000",      # 60s — tiny on purpose (size to consumer downtime in prod!)
    "segment.ms": "1000",         # roll the segment fast so retention has eligible segments
    "retention.bytes": "1048576", # 1 MB cap as a secondary trigger
})
produce_events(EVENTS_TOPIC, 500, value_fn=lambda i: {"event_id": i, "v": i * 1.0})

ends = topic_end_offsets(EVENTS_TOPIC)
print("end offsets (producer high-water mark):", ends)
print("produced 500 events to", EVENTS_TOPIC, "with a 60s retention / 1s segment")

## 1. Break it — (B) `compact` policy & a changelog of repeated keys

We create `kaf4_state` with **`cleanup.policy=compact`** plus the knobs that make compaction
*eligible* quickly at small scale (a tiny `segment.ms` and a very low `min.cleanable.dirty.ratio`).
Then we **produce many values for the same few keys** — `user-0` gets `v0..v9`, `user-1` gets
`v0..v9`, etc. The log physically holds *every* update; the **intended** end state is just the
**latest value per key**.

In [ ]:
ensure_topic(STATE_TOPIC, num_partitions=1, recreate=True, configs={
    "cleanup.policy": "compact",
    "segment.ms": "1000",                 # roll fast so segments become cleanable
    "min.cleanable.dirty.ratio": "0.01",  # clean even a barely-dirty log
    "delete.retention.ms": "1000",        # how long tombstones (deletes) linger
})

N_KEYS, N_VERSIONS = 5, 10                 # 5 keys x 10 updates = 50 records, but only 5 'current' values
def state_key(i):   return f"user-{i % N_KEYS}"
def state_value(i): return {"key": f"user-{i % N_KEYS}", "version": i // N_KEYS, "v": float(i)}
produce_events(STATE_TOPIC, N_KEYS * N_VERSIONS, key_fn=state_key, value_fn=state_value)

print(f"produced {N_KEYS*N_VERSIONS} records to {STATE_TOPIC} "
      f"({N_KEYS} keys x {N_VERSIONS} versions). End offsets:", topic_end_offsets(STATE_TOPIC))

## 2. Detect it — the configs are applied (`describe_configs`)

The first, rigorous proof: ask the broker what each topic's policy actually is. This is independent
of *when* the broker's cleanup threads run — it shows the topic **will** be retained / compacted as
configured. The same values appear in **kafka-ui → topic → Settings**.

In [ ]:
for t in (EVENTS_TOPIC, STATE_TOPIC):
    cfg = topic_config(t, SHOW)
    print(f"{t:<14}", {k: v for k, v in cfg.items() if v is not None})

ev = topic_config(EVENTS_TOPIC, ["cleanup.policy"])["cleanup.policy"]
st = topic_config(STATE_TOPIC,  ["cleanup.policy"])["cleanup.policy"]
assert ev == "delete" and st == "compact", (ev, st)
print("\nverified: kaf4_events is 'delete', kaf4_state is 'compact'")

## 3. Diagnose

**`delete`.** The log is a sequence of **segments**. A segment becomes deletable once it is
**rolled** (no longer active — `segment.ms`/`segment.bytes`) **and** older than `retention.ms` (or the
partition exceeds `retention.bytes`). A background thread deletes whole segments and **advances the
log-start (earliest) offset**. A consumer asking for an offset **below** the new log-start gets
**`OffsetOutOfRange`** — the data is gone, and `auto_offset_reset` is the *only* thing that decides
recovery.

**`compact`.** The log cleaner periodically rewrites **rolled** segments, keeping only the **latest
record per key** (a `null` value is a **tombstone** = delete marker, kept briefly via
`delete.retention.ms`). The **active** segment is never compacted, and a segment is only cleaned once
its dirty ratio exceeds `min.cleanable.dirty.ratio`. So compaction is **eventual**.

**Shared truth:** both are **broker-scheduled background jobs**. We set policy + thresholds (provable
now) and observe the effect later. Below we prove the semantics; the *physical* cleanup is on the
broker's clock.

## 3a. Compaction *intent* — read back & reduce to latest-per-key

We read the whole `kaf4_state` topic from the beginning and reduce to the **latest value per key** —
exactly the end state compaction converges to. Right after producing, the log still holds **all 50**
records; the broker will physically drop the superseded versions on its own schedule. We poll the
record count a few times (short, bounded) to show movement *if* the cleaner has already run — but we
do **not** block waiting for it.

In [ ]:
def read_all(topic, max_records=10_000, timeout_ms=3000):
    """Bounded read of a whole topic from the earliest AVAILABLE offset. Returns list of (key, value-bytes)."""
    c = KafkaConsumer(topic, bootstrap_servers=BOOTSTRAP, group_id=None,
                      auto_offset_reset="earliest", enable_auto_commit=False,
                      consumer_timeout_ms=timeout_ms,
                      key_deserializer=lambda b: b.decode() if b else None,
                      value_deserializer=lambda b: b)
    out = []
    try:
        for msg in c:
            out.append((msg.key, msg.value))
            if len(out) >= max_records:
                break
    finally:
        c.close()
    return out

records = read_all(STATE_TOPIC)
latest_per_key = {}
for k, v in records:                       # later offsets overwrite earlier -> 'latest wins'
    latest_per_key[k] = v
print(f"records physically present now : {len(records)}  (all versions until the cleaner runs)")
print(f"distinct keys                  : {sorted(latest_per_key)}")
print(f"compaction TARGET (latest/key) : {len(latest_per_key)} records — what compaction converges to")

In [ ]:
# Broker-scheduled: poll the record count a FEW times (short, bounded) to see if the cleaner ran.
# This may show no change on a fresh broker — that is expected and NOT a failure.
counts = []
for _ in range(3):
    counts.append(len(read_all(STATE_TOPIC)))
    time.sleep(2)                          # short, bounded — never an unbounded wait
print("physical record count over ~6s of polling:", counts)
print(f"target after full compaction would be {len(latest_per_key)} (latest-per-key).")
print("NOTE: collapse is broker-scheduled; if counts didn't drop, the cleaner simply hasn't run yet.")

## 3b. The `delete` recovery path — earliest *available* offset & `auto_offset_reset`

The offline-consumer scenario: a consumer committed offset *N* on `kaf4_events`, was down longer than
`retention.ms`, and comes back to request *N* — but retention has truncated the start, so the earliest
surviving offset is **> N**. The broker raises **`OffsetOutOfRange`**, and `auto_offset_reset` decides
recovery. We demonstrate the reliably-observable half: a fresh `auto_offset_reset="earliest"` consumer
is handed the **earliest offset that still exists** (which, after truncation, is exactly where a
reset-to-earliest consumer resumes). `"latest"` would instead jump to the end and **silently skip** the
gap.

In [ ]:
def first_available_offset(topic, reset):
    """Assign partition 0, seek to the beginning per `reset`, and report the first available offset."""
    c = KafkaConsumer(bootstrap_servers=BOOTSTRAP, group_id=None,
                      auto_offset_reset=reset, enable_auto_commit=False,
                      consumer_timeout_ms=3000, value_deserializer=lambda b: b)
    from kafka import TopicPartition
    tp = TopicPartition(topic, 0)
    c.assign([tp])
    try:
        if reset == "earliest":
            c.seek_to_beginning(tp)
        else:
            c.seek_to_end(tp)
        return c.position(tp)
    finally:
        c.close()

end = sum(topic_end_offsets(EVENTS_TOPIC).values())
earliest = first_available_offset(EVENTS_TOPIC, "earliest")
latest   = first_available_offset(EVENTS_TOPIC, "latest")
print(f"earliest AVAILABLE offset (auto_offset_reset='earliest'): {earliest}")
print(f"end offset               (auto_offset_reset='latest')  : {latest}")
print(f"\nA consumer reset to 'earliest' resumes at {earliest} (the oldest SURVIVING record after any")
print(f"retention truncation); 'latest' would jump to {latest} and skip everything in between.")
print("In prod, requesting a committed offset BELOW the earliest available is what raises OffsetOutOfRange.")

## 4. Fix it / guidance

| Need | Policy & settings | Why |
|------|-------------------|-----|
| Event log you replay within a known window | `cleanup.policy=delete`; size **`retention.ms`** ≥ worst-case consumer downtime + replay; optionally cap with **`retention.bytes`** | the offset you need still exists → no `OffsetOutOfRange` |
| Current-state / changelog (KTable, offsets, config) | `cleanup.policy=compact` (or `compact,delete`) | keeps latest-per-key without unbounded growth; rebuild state from offset 0 |
| Recover a consumer past retention | set **`auto_offset_reset`** on purpose — `earliest` to reprocess (idempotent sink, KAF-2), `latest` only if skipping the gap is OK | turns an unrecoverable error into defined behavior |
| Never age out | monitor **consumer lag** (KAF-2); alert before lag nears the retention window | retention failures are lag failures that went unnoticed |

## 5. Prove it

Pulling the three proofs together: (1) configs are applied (hard, immediate), (2) compaction's
latest-per-key target (semantics), (3) the earliest-available-offset recovery behavior.

In [ ]:
print("=== (1) configs applied (describe_configs) ===")
print(f"{EVENTS_TOPIC:<14}", {k: v for k, v in topic_config(EVENTS_TOPIC, SHOW).items() if v is not None})
print(f"{STATE_TOPIC:<14}", {k: v for k, v in topic_config(STATE_TOPIC, SHOW).items() if v is not None})

print("\n=== (2) compaction intent (latest-per-key) ===")
print(f"produced {len(records)} versioned records -> compaction target {len(latest_per_key)} "
      f"keys: {sorted(latest_per_key)}")

print("\n=== (3) delete-policy recovery (earliest available offset) ===")
print(f"earliest available = {earliest}, end = {latest}  "
      f"(a stale consumer below earliest -> OffsetOutOfRange; reset decides recovery)")
print("\nReminder: physical deletion & compaction are BROKER-SCHEDULED; the proofs above are the")
print("config + semantics you control, not an instant on-demand cleanup.")

## Takeaways & "in real production…"

- **Retention is a contract with your consumers.** Size `retention.ms` / `retention.bytes` to the
  *longest* a consumer might be down plus replay needs — not the default. Too small = silent loss for
  anyone who falls behind.
- **`OffsetOutOfRange` means "I fell past retention."** Set `auto_offset_reset` deliberately
  (`earliest` to reprocess with an idempotent sink, `latest` only when skipping is fine) and **alert**
  on it.
- **Compaction is for state, not history.** Use `cleanup.policy=compact` for changelog/KTable/offset
  topics (latest-per-key, bounded size, no time window); `compact,delete` adds a time ceiling;
  tombstones (`null`) propagate deletes.
- **Both cleanups are background-scheduled.** Never assume data vanished or collapsed the instant you
  changed a config — assert on the *config* and *semantics*; treat physical cleanup as eventual (tune
  with `segment.ms`, `min.cleanable.dirty.ratio`, `log.retention.check.interval.ms`).
- **Retention failures are lag failures.** A healthy, monitored consumer (KAF-2) never ages out; page
  on lag approaching the retention window.

## Teardown

In [ ]:
delete_topic(EVENTS_TOPIC)
delete_topic(STATE_TOPIC)
print("Deleted", EVENTS_TOPIC, "and", STATE_TOPIC, "(`make clean` also clears any local state).")